In [26]:
#Step 1) Loading a test file 
from langchain_community.document_loaders import TextLoader
loader = TextLoader("sample_text.txt")
documents = loader.load()
documents

[Document(metadata={'source': 'sample_text.txt'}, page_content="The Country That Accidentally Invented Itself\nIn 1847, a group of freed American slaves landed on the west coast of Africa and declared a republic. They wrote a constitution modelled almost word for word on the American one. They named their capital city Monrovia, after the American president James Monroe. They called their country Liberia — from the Latin word for freedom.\nThey had never been to Africa before. Most of them had been born in Virginia or Georgia or the Carolinas. They spoke English. They were Baptist and Methodist Christians. Their food, their music, their customs, their entire cultural framework was American — specifically, American Southern. Africa was not their homeland in any living memory. It was an idea, a symbol, a place their grandparents or great-grandparents had been taken from generations ago.\nAnd now they were colonists.\nThis is the uncomfortable paradox at the heart of Liberia's founding. Th

In [27]:
# Step 2) Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50) 
chunks = splitter.split_documents(documents)
chunks

[Document(metadata={'source': 'sample_text.txt'}, page_content='The Country That Accidentally Invented Itself'),
 Document(metadata={'source': 'sample_text.txt'}, page_content='In 1847, a group of freed American slaves landed on the west coast of Africa and declared a'),
 Document(metadata={'source': 'sample_text.txt'}, page_content='landed on the west coast of Africa and declared a republic. They wrote a constitution modelled'),
 Document(metadata={'source': 'sample_text.txt'}, page_content='a republic. They wrote a constitution modelled almost word for word on the American one. They named'),
 Document(metadata={'source': 'sample_text.txt'}, page_content='word for word on the American one. They named their capital city Monrovia, after the American'),
 Document(metadata={'source': 'sample_text.txt'}, page_content='their capital city Monrovia, after the American president James Monroe. They called their country'),
 Document(metadata={'source': 'sample_text.txt'}, page_content='president

#### First Run this to up Ollama

$ curl -fsSL https://ollama.com/install.sh | sh


$ ollama serve &

In [ ]:
# Step 3) Ollama Embeddings
from langchain_ollama import OllamaEmbeddings

#model is a required field
embeddings = OllamaEmbeddings(model='nomic-embed-text')

r1 = embeddings.embed_documents([chunk.page_content for chunk in chunks])
r1

[[0.10314731,
  0.047347497,
  -0.16836585,
  -0.025537442,
  0.08007863,
  -0.020787444,
  0.044001628,
  0.0022323248,
  -0.027277354,
  -0.013086083,
  -0.03150506,
  0.02096434,
  0.07130466,
  0.049653172,
  0.014809731,
  -0.051695198,
  0.021344205,
  -0.0032036204,
  -0.045990374,
  0.05199737,
  -0.036177784,
  -0.022730717,
  0.07081473,
  -0.026155997,
  -0.001028153,
  0.12525816,
  -0.027995486,
  -0.034842618,
  -0.025609253,
  -0.0072599724,
  0.074282184,
  -0.023078084,
  -0.019706968,
  -0.091481864,
  -0.06769677,
  -0.028075766,
  0.005868214,
  -0.019505765,
  0.05032037,
  -0.02154094,
  -0.026283063,
  -0.031524386,
  0.0041009486,
  -0.052595776,
  0.02766547,
  -0.024176704,
  -0.019297985,
  0.0076038362,
  0.031517316,
  -0.012156773,
  0.01358522,
  -0.02667903,
  -0.012949791,
  -0.031726267,
  0.0551102,
  -0.024323419,
  -0.032755885,
  -0.0036676913,
  0.058879778,
  -0.047004692,
  0.049175218,
  0.0096737705,
  0.018751934,
  0.011898383,
  0.032541994

In [21]:
len(r1[0])

768

In [28]:
#store in vector database
from langchain_chroma import Chroma

vector_database = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [ ]:
# Similarity Search

query = "What caused the financial crisis?"
results = vector_database.similarity_search(query)
results

[Document(id='cfc14180-a66b-4e7d-af8c-09116c0ea195', metadata={'source': 'sample_text.txt'}, page_content='human stories get told. The financial crisis was caused by one decision, one bank, one person. The'),
 Document(id='5f060943-6248-434d-b8de-e95161f48a28', metadata={'source': 'sample_text.txt'}, page_content='because that is how human memory works and how human stories get told. The financial crisis was'),
 Document(id='8a30931b-fe95-4544-8ad0-9583fa4e36bf', metadata={'source': 'sample_text.txt'}, page_content='caused by one decision, one bank, one person. The fall of Rome happened in 476 AD. The scientific'),
 Document(id='676c3a57-fd7c-43a1-a14b-0e6a5cbbba89', metadata={'source': 'sample_text.txt'}, page_content='distributed, slow-moving failures and compress them into single moments with identifiable causes')]

In [31]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = OllamaLLM(model="tinyllama")
retriever = vector_database.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke("What caused the 2008 financial crisis?")
print(response)

Based only on the given context, the question is asking for a single answer to the question "What caused the 2008 financial crizi?"
